# Generate Predictions with Threshold-based Retrieval

**Mục tiêu**: Tạo predictions cho evaluation, lưu **TẤT CẢ** items có similarity >= threshold (không chỉ top-10)

**Quy trình**:
1. Load 200 query_id từ ground truth JSONL
2. Load tất cả models (TFIDF, Ingredient_TFIDF, Keyword, Hybrid)
3. Với mỗi query_id:
   - Tính similarity scores với tất cả items trong candidate pool (10k recipes)
   - Lọc items có similarity >= threshold (exclude self)
   - Sắp xếp theo score giảm dần
4. Lưu predictions vào `evaluation_jsonl/<method>_pred.jsonl` với format:
   ```json
   {"query_id": 123, "relevant_docs": [{"doc_id": 456, "score": 0.85}, ...]}
   ```

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sentence_transformers import SentenceTransformer
import faiss

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup paths
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv"
MODELS_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models"
GROUND_TRUTH_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl"
OUTPUT_DIR = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl"

# Similarity threshold for relevant items
SIMILARITY_THRESHOLD = 0.1  # Items with similarity >= 0.1 are considered relevant
# Set to 0.0 to include all items with positive similarity

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Models path: {MODELS_PATH}")
print(f"Ground truth path: {GROUND_TRUTH_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")

Data path: E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv
Models path: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models
Ground truth path: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl
Output directory: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl
Similarity threshold: 0.1


## 1. Load Ground Truth and Data

In [3]:
# Ground truth (json)
ground_truth = []
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ground_truth.append(json.loads(line.strip()))

In [4]:
# query_ids (list)
query_ids = [item['query_id'] for item in ground_truth]
print(f"Loaded {len(query_ids)} query_ids from ground truth")
print(f"First 5 query_ids: {query_ids[:5]}")

Loaded 200 query_ids from ground truth
First 5 query_ids: [6302, 3779, 768, 3399, 4561]


In [5]:
# Load data (all recipes - 10k candidate pool)
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

Loaded 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


In [6]:
# Add recipe_id column (using index as recipe_id)
df['recipe_id'] = df.index
print(f"Columns: {df.columns.tolist()}")

Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source', 'recipe_id']


## 2. Load All Models

In [7]:
# Method 1: TFIDF (text-based: title + description + steps)
with open(os.path.join(MODELS_PATH, "TFIDF", "tfidf_vectorizer.pkl"), 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
    
tfidf_similarity = np.load(os.path.join(MODELS_PATH, "TFIDF", "tfidf_similarity.npy"))

with open(os.path.join(MODELS_PATH, "TFIDF", "tfidf_processed_data.pkl"), 'rb') as f:
    tfidf_data = pickle.load(f)

print(f"TFIDF similarity matrix shape: {tfidf_similarity.shape}")
print(f"TFIDF data shape: {len(tfidf_data)}")

TFIDF similarity matrix shape: (10263, 10263)
TFIDF data shape: 10263


In [8]:
# Method 2: Ingredient_TFIDF (ingredient-based)
with open(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_vectorizer.pkl"), 'rb') as f:
    ingredient_tfidf_vectorizer = pickle.load(f)
    
ingredient_tfidf_similarity = np.load(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_similarity.npy"))

print(f"Ingredient TFIDF similarity matrix shape: {ingredient_tfidf_similarity.shape}")

Ingredient TFIDF similarity matrix shape: (10263, 10263)


In [9]:
# Method 3: Keyword (TF-IDF on keywords extracted from title)
keyword_similarity = np.load(os.path.join(MODELS_PATH, "Keyword", "keyword_similarity.npy"))

print(f"Keyword similarity matrix shape: {keyword_similarity.shape}")

Keyword similarity matrix shape: (10263, 10263)


In [10]:
# Method 4: Hybrid (combine text_tfidf + ingredient_tfidf)
hybrid_similarity = np.load(os.path.join(MODELS_PATH, "Hybrid", "hybrid_similarity.npy"))

print(f"Hybrid similarity matrix shape: {hybrid_similarity.shape}")

Hybrid similarity matrix shape: (10263, 10263)


In [11]:
# Method 5: SBERT + FAISS (Semantic embeddings)

# Load model info
sbert_dir = os.path.join(MODELS_PATH, "SBERT_FAISS")
with open(os.path.join(sbert_dir, "model_info.json"), 'r', encoding='utf-8') as f:
    sbert_info = json.load(f)

# Load SBERT model
sbert_model = SentenceTransformer(sbert_info['model_name'])
print(f"  Loaded SBERT model: {sbert_info['model_name']}")

# Load recipe embeddings
sbert_embeddings = np.load(os.path.join(sbert_dir, "recipe_embeddings.npy"))
print(f"  SBERT embeddings shape: {sbert_embeddings.shape}")

# Load FAISS index
faiss_index = faiss.read_index(os.path.join(sbert_dir, "faiss_index.bin"))
print(f"  FAISS index loaded: {faiss_index.ntotal} vectors")

  Loaded SBERT model: keepitreal/vietnamese-sbert
  SBERT embeddings shape: (10263, 768)
  FAISS index loaded: 10263 vectors


In [12]:
# Compute SBERT similarity matrix (for threshold-based retrieval)
print("Computing SBERT similarity matrix...")
sbert_similarity = cosine_similarity(sbert_embeddings, sbert_embeddings)
print(f"  SBERT similarity matrix shape: {sbert_similarity.shape}")

Computing SBERT similarity matrix...
  SBERT similarity matrix shape: (10263, 10263)


In [13]:
# Method 6: Hybrid TF-IDF + SBERT (Ensemble)
print("Loading Hybrid TF-IDF + SBERT model...")

# Load config
hybrid_dir = os.path.join(MODELS_PATH, "Hybrid_TFIDF_SBERT")
with open(os.path.join(hybrid_dir, "config.json"), 'r', encoding='utf-8') as f:
    hybrid_config = json.load(f)

# Load SBERT embeddings (already loaded above, but load from hybrid dir for completeness)
hybrid_sbert_embeddings = np.load(os.path.join(hybrid_dir, "sbert_embeddings.npy"))
print(f"  Hybrid SBERT embeddings shape: {hybrid_sbert_embeddings.shape}")

# TF-IDF components are already loaded (reusing from TFIDF method)
print(f"  Reusing TF-IDF from method 1")
print(f"  Alpha (weight for TF-IDF): {hybrid_config['alpha']}")

# Compute hybrid similarity matrix (alpha * TF-IDF + (1-alpha) * SBERT)
alpha = hybrid_config['alpha']
print(f"Computing Hybrid similarity matrix (alpha={alpha})...")

# Normalize TF-IDF similarity to [0, 1] range
tfidf_sim_normalized = tfidf_similarity.copy()
# TF-IDF cosine similarity is already in [-1, 1], typically [0, 1] for non-negative vectors

# SBERT similarity is already normalized (cosine on normalized embeddings)
# Combine: alpha * TF-IDF + (1-alpha) * SBERT
hybrid_tfidf_sbert_similarity = alpha * tfidf_sim_normalized + (1 - alpha) * sbert_similarity
print(f"  Hybrid similarity matrix shape: {hybrid_tfidf_sbert_similarity.shape}")

Loading Hybrid TF-IDF + SBERT model...
  Hybrid SBERT embeddings shape: (10263, 768)
  Reusing TF-IDF from method 1
  Alpha (weight for TF-IDF): 0.5
Computing Hybrid similarity matrix (alpha=0.5)...
  Hybrid similarity matrix shape: (10263, 10263)


## 3. Implement Threshold-based Retrieval Functions

In [14]:
def retrieve_by_threshold(query_idx, similarity_matrix, threshold=0.1, exclude_self=True):
    """
    Args:
        query_idx: index of query item in dataframe
        similarity_matrix: precomputed similarity matrix
        threshold: minimum similarity score to be considered relevant
        exclude_self: whether to exclude query item itself
    
    Returns:
        list of (doc_id, score) tuples, sorted by score descending
    """
    # Get similarity scores for query
    scores = similarity_matrix[query_idx].copy()
    
    # Exclude self if requested
    if exclude_self:
        scores[query_idx] = -1  # Set to -1 to exclude from results
    
    # Find all indices with score >= threshold
    relevant_indices = np.where(scores >= threshold)[0]
    
    # Sort by score descending
    sorted_indices = relevant_indices[np.argsort(scores[relevant_indices])[::-1]]
    
    # Get doc_ids and scores
    results = []
    for idx in sorted_indices:
        doc_id = df.iloc[idx]['recipe_id']
        score = float(scores[idx])
        results.append({"doc_id": int(doc_id), "score": score})
    
    return results

In [15]:
# Test function with different thresholds
test_query_idx = 0
print(f"Test query (index {test_query_idx}):")
print(f"Query recipe_id: {df.iloc[test_query_idx]['recipe_id']}")
print(f"Query title: {df.iloc[test_query_idx]['title']}")
print()

Test query (index 0):
Query recipe_id: 0
Query title: Cách muối dưa hành truyền thống



In [16]:
for thresh in [0.5, 0.3, 0.1, 0.05]:
    test_results = retrieve_by_threshold(test_query_idx, tfidf_similarity, threshold=thresh)
    print(f"Threshold = {thresh}: Found {len(test_results)} relevant items")
    if len(test_results) > 0:
        print(f"  Top 3: {test_results[:3]}")
print()

Threshold = 0.5: Found 1 relevant items
  Top 3: [{'doc_id': 52, 'score': 0.6395764849733627}]
Threshold = 0.3: Found 10 relevant items
  Top 3: [{'doc_id': 52, 'score': 0.6395764849733627}, {'doc_id': 9321, 'score': 0.4543578750013707}, {'doc_id': 10057, 'score': 0.3689254056400364}]
Threshold = 0.1: Found 4666 relevant items
  Top 3: [{'doc_id': 52, 'score': 0.6395764849733627}, {'doc_id': 9321, 'score': 0.4543578750013707}, {'doc_id': 10057, 'score': 0.3689254056400364}]
Threshold = 0.05: Found 9601 relevant items
  Top 3: [{'doc_id': 52, 'score': 0.6395764849733627}, {'doc_id': 9321, 'score': 0.4543578750013707}, {'doc_id': 10057, 'score': 0.3689254056400364}]



In [17]:
# Use default threshold
test_results = retrieve_by_threshold(test_query_idx, tfidf_similarity, threshold=SIMILARITY_THRESHOLD)
print(f"Using threshold = {SIMILARITY_THRESHOLD}: Found {len(test_results)} relevant items")
print(f"Sample results (top 5):")
for i, item in enumerate(test_results[:5], 1):
    doc_row = df[df['recipe_id'] == item['doc_id']].iloc[0]
    print(f"  {i}. Doc {item['doc_id']} (score: {item['score']:.4f}) - {doc_row['title']}")

Using threshold = 0.1: Found 4666 relevant items
Sample results (top 5):
  1. Doc 52 (score: 0.6396) - Cách muối hành trắng giòn, để được lâu
  2. Doc 9321 (score: 0.4544) - Cách muối dưa hành giòn ngon, không bị hăng đơn giản tại nhà
  3. Doc 10057 (score: 0.3689) - Cách muối dưa củ cải giòn ngon không hăng bằng hộp đựng thực phẩm
  4. Doc 8290 (score: 0.3686) - Cách ngâm hành tím thái lát ăn liền ăn bún bò Huế chuẩn vị
  5. Doc 619 (score: 0.3302) - Dưa góp kiểu miền Trung


## 4. Run Evaluation Loop for All Methods

In [18]:
def run_evaluation(method_name, similarity_matrix, query_ids, df, threshold=0.1):
    """
    Run evaluation for a specific method using threshold-based retrieval
    
    Args:
        method_name: name of the method (for output file)
        similarity_matrix: precomputed similarity matrix
        query_ids: list of query recipe_ids
        df: dataframe containing all recipes
        threshold: minimum similarity score to consider an item relevant
    
    Returns:
        list of predictions (one per query)
    """
    predictions = []
    
    # Create recipe_id to index mapping
    recipe_id_to_idx = {recipe_id: idx for idx, recipe_id in enumerate(df['recipe_id'])}
    
    # Track statistics
    total_relevant = 0
    min_relevant = float('inf')
    max_relevant = 0
    
    for query_id in tqdm(query_ids, desc=f"Evaluating {method_name}"):
        # Get query index
        query_idx = recipe_id_to_idx[query_id]
        
        # Retrieve ALL items >= threshold (excluding self)
        relevant_items = retrieve_by_threshold(
            query_idx, 
            similarity_matrix, 
            threshold=threshold, 
            exclude_self=True
        )
        
        # Update statistics
        num_relevant = len(relevant_items)
        total_relevant += num_relevant
        min_relevant = min(min_relevant, num_relevant)
        max_relevant = max(max_relevant, num_relevant)
        
        # Create prediction record
        pred_record = {
            "query_id": int(query_id),
            "relevant_docs": relevant_items  # All items >= threshold, not just top-10
        }
        predictions.append(pred_record)
    
    # Print statistics
    avg_relevant = total_relevant / len(query_ids)
    print(f"\nStatistics for {method_name}:")
    print(f"  Average relevant items per query: {avg_relevant:.2f}")
    print(f"  Min relevant items: {min_relevant}")
    print(f"  Max relevant items: {max_relevant}")
    print(f"  Total relevant pairs: {total_relevant}")
    
    return predictions

In [19]:
# Test with small subset first
test_predictions = run_evaluation("test", tfidf_similarity, query_ids[:5], df, threshold=SIMILARITY_THRESHOLD)
print(f"\nExample prediction:")
print(f"Query ID: {test_predictions[0]['query_id']}")
print(f"Number of relevant docs: {len(test_predictions[0]['relevant_docs'])}")
print(f"Top 5 relevant docs: {test_predictions[0]['relevant_docs'][:5]}")

Evaluating test: 100%|██████████| 5/5 [00:00<00:00,  5.29it/s]


Statistics for test:
  Average relevant items per query: 3567.00
  Min relevant items: 2232
  Max relevant items: 5370
  Total relevant pairs: 17835

Example prediction:
Query ID: 6302
Number of relevant docs: 4730
Top 5 relevant docs: [{'doc_id': 6547, 'score': 0.7862034845555061}, {'doc_id': 6605, 'score': 0.7484998247435155}, {'doc_id': 6160, 'score': 0.7422538990467994}, {'doc_id': 7121, 'score': 0.6669771498939066}, {'doc_id': 4790, 'score': 0.6642594310179872}]


In [20]:
# Run evaluation for all 6 methods with threshold-based retrieval
methods = [
    ("TFIDF", tfidf_similarity),
    ("Ingredient_TFIDF", ingredient_tfidf_similarity),
    ("Keyword", keyword_similarity),
    ("Hybrid", hybrid_similarity),
    ("SBERT_FAISS", sbert_similarity),
    ("Hybrid_TFIDF_SBERT", hybrid_tfidf_sbert_similarity)
]

all_predictions = {}

print(f"Running evaluation with THRESHOLD = {SIMILARITY_THRESHOLD}")
print(f"This will retrieve ALL items with similarity >= {SIMILARITY_THRESHOLD}")
print(f"Total methods: {len(methods)}")

Running evaluation with THRESHOLD = 0.1
This will retrieve ALL items with similarity >= 0.1
Total methods: 6


In [21]:
for method_name, similarity_matrix in methods:
    print(f"\n{'='*80}")
    print(f"Method: {method_name}")
    print(f"{'='*80}")
    
    # Run evaluation with threshold
    predictions = run_evaluation(
        method_name, 
        similarity_matrix, 
        query_ids, 
        df, 
        threshold=SIMILARITY_THRESHOLD
    )
    all_predictions[method_name] = predictions
    
    # Save to JSONL file
    output_file = os.path.join(OUTPUT_DIR, f"{method_name}_pred.jsonl")
    with open(output_file, 'w', encoding='utf-8') as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + '\n')
    
    print(f"Saved {len(predictions)} predictions to {output_file}")
    print(f"  Total relevant pairs: {sum(len(p['relevant_docs']) for p in predictions)}")


Method: TFIDF


Evaluating TFIDF: 100%|██████████| 200/200 [00:34<00:00,  5.74it/s]



Statistics for TFIDF:
  Average relevant items per query: 3895.97
  Min relevant items: 196
  Max relevant items: 8506
  Total relevant pairs: 779195
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\TFIDF_pred.jsonl
  Total relevant pairs: 779195

Method: Ingredient_TFIDF


Evaluating Ingredient_TFIDF: 100%|██████████| 200/200 [00:16<00:00, 11.78it/s]



Statistics for Ingredient_TFIDF:
  Average relevant items per query: 1839.74
  Min relevant items: 148
  Max relevant items: 4349
  Total relevant pairs: 367949
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\Ingredient_TFIDF_pred.jsonl
  Total relevant pairs: 367949

Method: Keyword


...


Statistics for Hybrid:
  Average relevant items per query: 2426.80
  Min relevant items: 167
  Max relevant items: 5599
  Total relevant pairs: 485360
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\Hybrid_pred.jsonl
  Total relevant pairs: 485360

Method: SBERT_FAISS


Evaluating SBERT_FAISS: 100%|██████████| 200/200 [01:32<00:00,  2.17it/s]



Statistics for SBERT_FAISS:
  Average relevant items per query: 10261.53
  Min relevant items: 10240
  Max relevant items: 10262
  Total relevant pairs: 2052306
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\SBERT_FAISS_pred.jsonl
  Total relevant pairs: 2052306

Method: Hybrid_TFIDF_SBERT


Evaluating Hybrid_TFIDF_SBERT: 100%|██████████| 200/200 [01:34<00:00,  2.12it/s]



Statistics for Hybrid_TFIDF_SBERT:
  Average relevant items per query: 10256.95
  Min relevant items: 10102
  Max relevant items: 10262
  Total relevant pairs: 2051391
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\Hybrid_TFIDF_SBERT_pred.jsonl
  Total relevant pairs: 2051391


In [22]:
# summary
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    total_pairs = sum(len(p['relevant_docs']) for p in all_predictions[method_name])
    avg_pairs = total_pairs / len(query_ids)
    print(f"  {method_name:25s}: {total_pairs:6d} total pairs ({avg_pairs:6.2f} avg per query)")

  TFIDF                    : 779195 total pairs (3895.97 avg per query)
  Ingredient_TFIDF         : 367949 total pairs (1839.74 avg per query)
  Keyword                  : 229400 total pairs (1147.00 avg per query)
  Hybrid                   : 485360 total pairs (2426.80 avg per query)
  SBERT_FAISS              : 2052306 total pairs (10261.53 avg per query)
  Hybrid_TFIDF_SBERT       : 2051391 total pairs (10256.95 avg per query)


## 6. Sanity Checks

### 6.1. Verify no self-recommendations

In [23]:
# Verify that no prediction contains the query_id itself
def verify_no_self_recommendations(predictions):
    """Check if any prediction contains self-recommendation"""
    violations = []
    
    for pred in predictions:
        query_id = pred['query_id']
        doc_ids = [item['doc_id'] for item in pred['relevant_docs']]
        
        if query_id in doc_ids:
            violations.append(query_id)
    
    return violations

In [24]:
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    preds = all_predictions[method_name]
    violations = verify_no_self_recommendations(preds)
    
    if violations:
        print(f"{method_name}: Found {len(violations)} self-recommendations!")
        print(f"   Query IDs with self-recommendation: {violations[:5]}...")
    else:
        print(f"{method_name}: No self-recommendations found ({len(preds)} queries checked)")

TFIDF: No self-recommendations found (200 queries checked)
Ingredient_TFIDF: No self-recommendations found (200 queries checked)
Keyword: No self-recommendations found (200 queries checked)
Hybrid: No self-recommendations found (200 queries checked)
SBERT_FAISS: No self-recommendations found (200 queries checked)
Hybrid_TFIDF_SBERT: No self-recommendations found (200 queries checked)


### 6.2. Check score distributions

In [25]:
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    preds = all_predictions[method_name]
    sizes = [len(p['relevant_docs']) for p in preds]
    
    print(f"\n{method_name}:")
    print(f"  Min relevant items: {min(sizes)}")
    print(f"  Max relevant items: {max(sizes)}")
    print(f"  Mean: {np.mean(sizes):.2f}")
    print(f"  Median: {np.median(sizes):.2f}")
    print(f"  Std: {np.std(sizes):.2f}")
    
    # Percentiles
    percentiles = [25, 50, 75, 90, 95, 99]
    print(f"  Percentiles:")
    for p in percentiles:
        val = np.percentile(sizes, p)
        print(f"    {p}th: {val:.0f}")


TFIDF:
  Min relevant items: 196
  Max relevant items: 8506
  Mean: 3895.97
  Median: 3721.50
  Std: 2094.06
  Percentiles:
    25th: 2332
    50th: 3722
    75th: 5668
    90th: 6802
    95th: 7306
    99th: 8090

Ingredient_TFIDF:
  Min relevant items: 148
  Max relevant items: 4349
  Mean: 1839.74
  Median: 1610.00
  Std: 1114.67
  Percentiles:
    25th: 896
    50th: 1610
    75th: 2742
    90th: 3500
    95th: 3750
    99th: 4162

Keyword:
  Min relevant items: 4
  Max relevant items: 4330
  Mean: 1147.00
  Median: 1025.00
  Std: 902.60
  Percentiles:
    25th: 330
    50th: 1025
    75th: 1791
    90th: 2435
    95th: 2677
    99th: 3276

Hybrid:
  Min relevant items: 167
  Max relevant items: 5599
  Mean: 2426.80
  Median: 2190.00
  Std: 1363.81
  Percentiles:
    25th: 1292
    50th: 2190
    75th: 3382
    90th: 4467
    95th: 4737
    99th: 5059

SBERT_FAISS:
  Min relevant items: 10240
  Max relevant items: 10262
  Mean: 10261.53
  Median: 10262.00
  Std: 2.31
  Percentiles

### 6.3. Score distribution analysis

In [26]:
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    preds = all_predictions[method_name]
    all_scores = []
    for pred in preds:
        all_scores.extend([item['score'] for item in pred['relevant_docs']])
    
    if len(all_scores) > 0:
        print(f"\n{method_name} (total {len(all_scores)} relevant pairs):")
        print(f"  Score range: [{min(all_scores):.4f}, {max(all_scores):.4f}]")
        print(f"  Mean: {np.mean(all_scores):.4f}")
        print(f"  Median: {np.median(all_scores):.4f}")
        
        # Count by threshold
        for thresh in [0.5, 0.3, 0.2, 0.1, 0.05]:
            count = sum(1 for s in all_scores if s >= thresh)
            pct = 100 * count / len(all_scores)
            print(f"  >= {thresh:.2f}: {count:6d} ({pct:5.2f}%)")


TFIDF (total 779195 relevant pairs):
  Score range: [0.1000, 0.9794]
  Mean: 0.1668
  Median: 0.1410
  >= 0.50:   6295 ( 0.81%)
  >= 0.30:  50953 ( 6.54%)
  >= 0.20: 159530 (20.47%)
  >= 0.10: 779195 (100.00%)
  >= 0.05: 779195 (100.00%)

Ingredient_TFIDF (total 367949 relevant pairs):
  Score range: [0.1000, 1.0000]
  Mean: 0.1633
  Median: 0.1459
  >= 0.50:    536 ( 0.15%)
  >= 0.30:  13435 ( 3.65%)
  >= 0.20:  75795 (20.60%)
  >= 0.10: 367949 (100.00%)
  >= 0.05: 367949 (100.00%)

Keyword (total 229400 relevant pairs):
  Score range: [0.1000, 0.5714]
  Mean: 0.1350
  Median: 0.1250
  >= 0.50:     15 ( 0.01%)
  >= 0.30:    885 ( 0.39%)
  >= 0.20:  14374 ( 6.27%)
  >= 0.10: 229400 (100.00%)
  >= 0.05: 229400 (100.00%)

Hybrid (total 485360 relevant pairs):
  Score range: [0.1000, 0.9918]
  Mean: 0.1584
  Median: 0.1409
  >= 0.50:    432 ( 0.09%)
  >= 0.30:  15701 ( 3.23%)
  >= 0.20:  86385 (17.80%)
  >= 0.10: 485360 (100.00%)
  >= 0.05: 485360 (100.00%)

SBERT_FAISS (total 2052306 re